# Track 07 — Autonomous Agentic RAG and DeepAgent Systems

Welcome to **Track 07** of the RAG Engineering Academy. In Track 05 and Track 06, we moved from declarative LCEL chains to cyclical state machines with LangGraph. However, deterministic graphs still rely on predefined edge routing.

When queries require non-deterministic multi-hop reasoning—such as searching unstructured text, querying an operational SQL database, calculating exact financial metrics, and iteratively refining until a goal is met—we require **Autonomous Agentic RAG**.

### What You Will Build & Master:
1. **Tool Abstractions & Permission Guards**: Type-safe tool definitions, JSON Schema generation, and permission tiers (`READ_ONLY`, `RESTRICTED`, `EXECUTE`).
2. **The ReAct Execution Loop**: First-principles implementation of Thought $\to$ Action $\to$ Observation $\to$ Reflection.
3. **Execution Budgets & Loop Guardrails**: Call signature hashing, iteration limits, and deterministic cycle detection.
4. **Plan-and-Solve Dynamic Planning**: Multi-hop query decomposition into dependency DAGs and dynamic replanning on step failure.
5. **DeepAgent Multi-Agent Coordination**: Supervisor-Specialist architecture with dedicated Retrieval Specialists, Data Analysts, and Fact-Checking Critics.
6. **Agent Benchmarking & Failure Taxonomy**: Quantitative metrics for tool precision, redundant call ratio, and grounded fact recall.



## 1. Paradigm Shift: Static RAG vs. Stateful Graphs vs. Autonomous Agents

In standard RAG, the query flow is fixed ($Q \to E \to R \to G$). In LangGraph (Track 06), we introduced cyclical states and conditional edges, but the graph transitions followed hardcoded branching logic.

In **Agentic RAG**, the model itself acts as an autonomous reasoning engine:
- The agent inspects available tools and decides *whether* retrieval is needed.
- It determines *which* tools to call, with *what* arguments, and in *what* sequence.
- It observes intermediate results and reflects on whether the user's goal is satisfied.

Let us inspect the foundational ReAct loop and loop guardrails architecture:


In [1]:
from IPython.display import Image, display
from pathlib import Path
import os
import sys

# Ensure repository root is on Python path
REPO_ROOT = Path('.').resolve().parent.parent if Path('.').resolve().name == '07-agentic-rag' else Path('.').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from common.config import settings
from common.agentic_rag import (
    ToolPermission,
    ToolDefinition,
    ToolRegistry,
    SafeCalculator,
    create_in_memory_database,
    create_default_tool_registry,
    AgentBudget,
    AgentStep,
    AgentTrajectory,
    ReActAgent,
    PlanStep,
    ExecutionPlan,
    PlanAndSolveEngine,
    RetrievalSpecialist,
    DataAnalystSpecialist,
    CriticSpecialist,
    DeepAgentCoordinator,
    AgentEvaluator,
    AgentFailureMode,
)

diagram_path = REPO_ROOT / 'assets' / 'diagrams' / 'track07_agentic_rag_react_loop.png'
if diagram_path.exists():
    display(Image(filename=str(diagram_path), width=850))
else:
    print('Diagram ready at:', diagram_path)



<IPython.core.display.Image object>


## 2. Tool Abstractions, Parameter Schemas, and Permission Gating

Autonomous agents are only as safe as the tools they are allowed to execute. A production agent must never be given unrestricted code execution or write permissions without strict sandboxing.

### Architectural Principles:
1. **Schema Explicitness**: Every tool exposes a JSON Schema (compatible with OpenAI and Groq function calling standards) defining parameter types and descriptions.
2. **Access Control Tiers**: Tools are tagged with permissions (`READ_ONLY`, `RESTRICTED`, `EXECUTE`, `ADMIN`). The registry enforces policies preventing lower-privilege agents from executing destructive tools.
3. **AST Sandboxing**: Math evaluation uses python's `ast` (Abstract Syntax Tree) module instead of unsafe `eval()`, preventing injection attacks.

Let us instantiate our tool registry with document retrieval, SQL database querying, safe calculation, and external search:


In [2]:
# Instantiate the default tool registry populated with in-memory knowledge and SQL databases
registry = create_default_tool_registry()

print("Registered Tools in Academy Registry:")
for tool in registry.list_tools():
    print(f"  - [{tool.permission.value.upper()}] {tool.name}: {tool.description[:65]}...")

# Inspect OpenAI/Groq function calling schema for the document retriever
schemas = registry.get_schemas()
print(f"\nSample Tool Schema ({schemas[0]['function']['name']}):")
import json
print(json.dumps(schemas[0], indent=2))



Registered Tools in Academy Registry:
  - [READ_ONLY] search_documents: Retrieve unstructured text documents, reports, and whitepapers us...
  - [READ_ONLY] query_database: Execute read-only SQL queries against corporate financial and ope...
  - [READ_ONLY] calculate: Safely evaluate mathematical expressions, percentages, growth rat...
  - [RESTRICTED] search_external_web: Search external market indexes and breaking news for external con...

Sample Tool Schema (search_documents):
{
  "type": "function",
  "function": {
    "name": "search_documents",
    "description": "Retrieve unstructured text documents, reports, and whitepapers using semantic keyword search.",
    "parameters": {
      "type": "object",
      "properties": {
        "query": {
          "type": "string",
          "description": "The search query string."
        },
        "top_k": {
          "type": "integer",
          "description": "Number of results to return."
        }
      },
      "required": [
        

## 3. The Autonomous ReAct Loop: Multi-Hop Reasoning Trajectory

The **ReAct** (Reason + Act) paradigm prompts the language model to interleave reasoning traces ("Thought") with action executions ("Action" and "Action Input"), receiving observations from the environment before formulating subsequent actions.

### The Problem:
Consider the complex business inquiry:
> *"Find NovaTech AI Q3 revenue and calculate its operating profit margin."*

A naive single-turn RAG pipeline cannot answer this because:
1. The text document mentions revenue ($61M) and operating expenses ($39M), but does not state the operating margin percentage.
2. The exact audited operating costs reside in the corporate relational database.
3. Language models are notorious for arithmetic inaccuracies when computing decimal division and percentages directly.

Let us execute the `ReActAgent` to observe how it autonomously calls the document retriever, cross-references the SQL database, computes the exact margin via `SafeCalculator`, and synthesizes a grounded response:


In [3]:
# Initialize ReAct Agent with execution budget
budget = AgentBudget(max_iterations=6, max_tool_calls=6)
agent = ReActAgent(tool_registry=registry, budget=budget)

query = "Find NovaTech AI Q3 revenue and calculate its operating profit margin."
trajectory = agent.run(query)

print(f"Execution Status: {trajectory.termination_reason} (Success={trajectory.success})")
print(f"Total Execution Time: {trajectory.total_execution_time_ms:.2f} ms\n")

print("--- STEP-BY-STEP ReAct REASONING TRAJECTORY ---")
for step in trajectory.steps:
    print(f"Step {step.step_number}:")
    print(f"  Thought: {step.thought}")
    if step.action:
        print(f"  Action: {step.action}({step.action_input})")
        print(f"  Observation: {step.observation.strip()[:120]}...")
    if step.reflection:
        print(f"  Reflection: {step.reflection}")
    print()

print("--- FINAL VERIFIED ANSWER ---")
print(trajectory.final_answer)



Execution Status: GOAL_COMPLETED (Success=True)
Total Execution Time: 0.22 ms

--- STEP-BY-STEP ReAct REASONING TRAJECTORY ---
Step 1:
  Thought: I need to locate NovaTech AI's reported Q3-2026 revenue and operating expenses in the knowledge base.
  Action: search_documents({'query': 'NovaTech AI Q3-2026 revenue operating expenses'})
  Observation: [
  {
    "id": "doc-01",
    "title": "NovaTech AI Q3 Shareholder Letter",
    "content": "NovaTech AI achieved record ...
  Reflection: Observed result from 'search_documents'. Data successfully acquired.

Step 2:
  Thought: To verify the exact figures, I should query the structured corporate finances table.
  Action: query_database({'sql_query': "SELECT revenue, operating_costs, net_income FROM corporate_finances WHERE company='NovaTech AI' AND quarter='Q3-2026'"})
  Observation: [
  {
    "revenue": 61000000.0,
    "operating_costs": 39000000.0,
    "net_income": 22000000.0
  }
]...
  Reflection: Observed result from 'query_database'. Da

## 4. Guardrails, Loop Prevention, and Call Signature Hashing

One of the most dangerous failure modes of autonomous agents is the **Infinite Invocation Loop**:
- An agent calls a tool, receives an ambiguous or partial answer, and reformulates the exact same query.
- It enters an infinite loop, consuming entire API budgets and locking server threads.

### Prevention Mechanism:
We implement **Call Signature Hashing**:
$$\text{sig} = \text{SHA256}(\text{ToolName} + \text{CanonicalJSON}(\text{Arguments}))$$

If an agent attempts to invoke the exact same tool with identical arguments more than `max_retries_per_tool` times, the budget guardrail trips, halting the execution and forcing reflection or termination.

Let us deliberately test this guardrail with a stuck agent policy:


In [4]:
# Simulate an agent with a stuck policy that repeats identical queries
def stuck_agent_policy(user_query, step_history):
    return ("I need to search again for the exact same term.", "search_documents", {"query": "infinite loop probe"}, None)

constrained_budget = AgentBudget(max_iterations=8, max_retries_per_tool=2)
stuck_agent = ReActAgent(tool_registry=registry, budget=constrained_budget)

aborted_trajectory = stuck_agent.run("Infinite Loop Test Query", offline_policy=stuck_agent_policy)

print(f"Termination Reason: {aborted_trajectory.termination_reason}")
print(f"Success Flag: {aborted_trajectory.success}")
print(f"Final Guardrail Message: {aborted_trajectory.final_answer}")
print(f"Steps Recorded Before Intervention: {len(aborted_trajectory.steps)}")



Termination Reason: LOOP_DETECTED
Success Flag: False
Final Guardrail Message: Execution aborted by guardrails: Loop detected: Tool 'search_documents' invoked 3 times with identical arguments.
Steps Recorded Before Intervention: 3


## 5. Dynamic Planning & Decomposition: Plan-and-Solve Architecture

While ReAct works well for step-by-step exploration, complex multi-agent workflows benefit from **Plan-and-Solve** (Wang et al., 2023):
1. **Planning Phase**: The agent decomposes a multifaceted user question into an ordered DAG of atomic sub-tasks with explicit dependencies.
2. **Execution Phase**: Each sub-task is executed sequentially, feeding intermediate context forward.
3. **Dynamic Replanning**: If an intermediate step fails (e.g. database query error or zero search results), the replanner dynamically inserts a fallback step.

Let us inspect the Plan-and-Solve architecture:


In [5]:
planner_diagram = REPO_ROOT / 'assets' / 'diagrams' / 'track07_plan_and_solve_multiagent.png'
if planner_diagram.exists():
    display(Image(filename=str(planner_diagram), width=850))

# Execute Plan-and-Solve Engine
planner = PlanAndSolveEngine(tool_registry=registry, max_replans=2)
comparison_query = "Compare NovaTech AI and Helios Dynamics Q3 financial performance"

plan = planner.formulate_plan(comparison_query)
print(f"Formulated Plan for: '{plan.goal}'")
for s in plan.steps:
    print(f"  Step {s.step_id}: {s.description} -> [{s.tool_name}]")

completed_plan, plan_summary = planner.execute_plan(plan)
print(f"\nPlan Completed: {completed_plan.is_complete} (Replans: {completed_plan.replan_count})")
print("\n--- EXECUTION SUMMARY ---")
print(plan_summary)



<IPython.core.display.Image object>
Formulated Plan for: 'Compare NovaTech AI and Helios Dynamics Q3 financial performance'
  Step 1: Retrieve NovaTech AI Q3 financials from structured database -> [query_database]
  Step 2: Retrieve Helios Dynamics Q3 financials from structured database -> [query_database]
  Step 3: Compute NovaTech AI revenue premium percentage over Helios Dynamics -> [calculate]

Plan Completed: True (Replans: 0)

--- EXECUTION SUMMARY ---
Execution Plan Completed for: 'Compare NovaTech AI and Helios Dynamics Q3 financial performance'
- Retrieve NovaTech AI Q3 financials from structured database: [
  {
    "revenue": 61000000.0,
    "net_income": 22000000.0
  }
]
- Retrieve Helios Dynamics Q3 financials from structured database: [
  {
    "revenue": 34000000.0,
    "net_income": 9000000.0
  }
]
- Compute NovaTech AI revenue premium percentage over Helios Dynamics: 79.41


## 6. DeepAgent Multi-Agent Coordination: Supervisor & Specialist Workers

In enterprise architectures, a monolithic agent handling everything often suffers from context pollution and tool confusion. **DeepAgent** decomposes the workflow into specialized micro-agents:

1. **Supervisor Agent**: Maintains the global objective, coordinates message passing, and synthesizes final output.
2. **Retrieval Specialist**: Focuses purely on dense semantic search, lexical queries, and chunk reranking.
3. **Data Analyst Specialist**: Interfaces with structured tabular databases and exact arithmetic calculators.
4. **Critic Specialist**: Acts as an independent fact-checker, calculating the grounding score between claims and retrieved evidence before approving output.

Let us execute the `DeepAgentCoordinator`:


In [6]:
# Initialize DeepAgent Multi-Agent Coordinator
coordinator = DeepAgentCoordinator(registry)

goal = "Analyze NovaTech AI Q3 operating margins and corporate health"
result = coordinator.solve(goal)

print(f"Mission Goal: {result['goal']}")
print(f"Critic Specialist Verdict: {result['critic_verdict']} (Grounding Score: {result['grounding_score']:.1%})")
print("\n--- INTER-AGENT COMMUNICATION TRACE ---")
for t in result['trace']:
    print(f"  {t}")

print("\n--- FINAL VERIFIED SYNTHESIS ---")
print(result['final_answer'])



Mission Goal: Analyze NovaTech AI Q3 operating margins and corporate health
Critic Specialist Verdict: PASS (Grounding Score: 100.0%)

--- INTER-AGENT COMMUNICATION TRACE ---
  Supervisor received goal: Analyze NovaTech AI Q3 operating margins and corporate health
  Supervisor dispatched task to RetrievalSpecialist.
  Supervisor dispatched task to DataAnalystSpecialist.
  Supervisor dispatched draft to CriticSpecialist for verification.
  Critic returned verdict: PASS (score: 1.0)

--- FINAL VERIFIED SYNTHESIS ---
NovaTech AI reported Q3 revenue of $61.0M and operating costs of $39.0M, resulting in operating income of $22,000,000.


## 7. Agent Evaluation & Failure Mode Taxonomy

How do we quantitatively benchmark autonomous RAG agents? We track four key performance dimensions:

| Metric | Definition | Ideal Target |
|---|---|---|
| **Tool Accuracy** | Harmonic mean ($F_1$) of tool selection precision and recall against optimal tools | $\ge 90\%$ |
| **Redundancy Ratio** | Fraction of identical duplicate tool calls: $1 - \frac{\text{Unique Calls}}{\text{Total Calls}}$ | $0.0\%$ |
| **Grounded Fact Recall** | Percentage of verifiable ground truth facts present in final answer | $100\%$ |
| **Budget Efficiency** | Steps utilized relative to the maximum iteration budget | $\le 60\%$ |

Let us evaluate our ReAct trajectory using the `AgentEvaluator`:


In [7]:
# Evaluate the ReAct trajectory generated in Section 3
eval_metrics = AgentEvaluator.evaluate_trajectory(
    trajectory=trajectory,
    expected_tools={"search_documents", "query_database", "calculate"},
    ground_truth_facts=["NovaTech AI", "61.0M", "39.0M"],
)

print("=== QUANTITATIVE AGENT BENCHMARK REPORT ===")
for metric, val in eval_metrics.items():
    print(f"  {metric:22s}: {val}")



=== QUANTITATIVE AGENT BENCHMARK REPORT ===
  success               : True
  failure_mode          : success
  tool_accuracy         : 1.0
  redundancy_ratio      : 0.0
  fact_recall           : 1.0
  step_count            : 4
  execution_time_ms     : 0.22


## 8. Optional Live Groq Tool Calling Integration

When `GROQ_API_KEY` is configured in your root `.env` file, the agent can connect directly to Groq's high-speed inference engine using standard function calling schemas.

The cell below checks for the presence of your API key:
- If present, you can run live inference with models like `llama-3.3-70b-versatile`.
- If absent, it gracefully demonstrates the schema payload that would be dispatched to the API.


In [8]:
# Inspect live Groq configuration and demonstrate tool calling payload
print(f"Groq API Key Configured: {settings.groq_api_key is not None}")

if settings.groq_api_key:
    print("Live Groq API key detected. Tools are ready for live Groq function calling!")
    try:
        from groq import Groq
        groq_client = Groq(api_key=settings.groq_api_key)
        print("Successfully connected to Groq client for agentic tool execution.")
    except Exception as e:
        print(f"Groq client note: {e}")
else:
    print("Running in 100% offline educational mode. No API key required for full curriculum completion.")



Groq API Key Configured: False
Running in 100% offline educational mode. No API key required for full curriculum completion.


## 9. Architectural Takeaways & Production Checklist

### Summary of Core Learnings:
1. **Autonomy Requires Guardrails**: Autonomous tool-calling without execution budgets and call signature hashing will inevitably encounter infinite loops or catastrophic quota consumption.
2. **Decomposition Beats Monolithic ReAct**: For complex queries spanning multiple data modalities (documents + databases), Plan-and-Solve and Multi-Agent delegation dramatically reduce cognitive load and tool confusion.
3. **Independent Criticism is Essential**: Having a separate Critic Specialist evaluate grounding before answering prevents hallucinated numbers from reaching end users.

### Production Readiness Checklist:
- [x] Strict tool permissions (`READ_ONLY`, `RESTRICTED`, `EXECUTE`) enforced at the registry level.
- [x] Safe AST-based calculator replacing unsafe code evaluation.
- [x] Call signature hashing (`SHA256`) preventing cyclical tool calling.
- [x] Step and time budgets enforced on every iteration.
- [x] Dynamic replanning capability handling intermediate step failures.
- [x] Quantitative evaluation metrics (Tool Precision, Redundancy Ratio, Fact Grounding).

In the next track (**Track 08 — Memory Engineering**), we will explore conversation state, episodic memory, entity consolidation, and forgetting policies in stateful multi-turn RAG systems!

